In [1]:
import sys
import google.protobuf

print("Python:", sys.executable)
print("Protobuf version:", google.protobuf.__version__)
print("Protobuf location:", google.protobuf.__file__)

Python: d:\Anaconda\python.exe
Protobuf version: 6.33.6
Protobuf location: d:\Anaconda\lib\site-packages\google\protobuf\__init__.py


In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Coding\\Development\\AI\\Medical Chatbot'

In [5]:
# %pip install langchain==0.3.26 flask==3.1.1 pypdf==5.6.1 sentence-transformers==4.1.0 python-dotenv==1.1.0 langchain-pinecone==0.2.8 langchain-openai==0.3.24 langchain-community==0.3.26

## Data Extraction

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
# Extract Text from PDF File
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files("data")

In [9]:
# extracted_data

In [10]:
len(extracted_data)

637

In [11]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containing only 'source' in metdata and the original page_content
    """

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [13]:
# minimal_docs

## Chunking (Split document into smaller chunks)

In [14]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [15]:
text_chunk = text_split(minimal_docs)
print(f"Number of Chunks: {len(text_chunk)}")

Number of Chunks: 5859


## Embedding Model

In [16]:
# %pip install --upgrade --force-reinstall numpy scikit-learn

In [17]:
# %pip install --force-reinstall "numpy==1.26.4" scikit-learn

In [ ]:
# from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embeddings model 
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embedding = download_embedding()

RuntimeError: Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [ ]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vector = embedding.embed_query("Hello World")
# vector

In [ ]:
print(f"Length of Vector: {len(vector)}")

Length of Vector: 384


## Pinecone Vector DB

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [ ]:
MEDICALBOT_PINECONE_API_KEY = os.getenv("MEDICALBOT_PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["MEDICALBOT_PINECONE_API_KEY"] = MEDICALBOT_PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = MEDICALBOT_PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=384, metric="cosine", spec=ServerlessSpec(cloud='aws', region="us-east-1"))

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)

Loading Existing Index

In [ ]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

Add More Data to the existing Pinecone index

In [ ]:
dummyDoc = Document(
    page_content="Hi this is Ninad More, a software engineer at Barclays",
    metadata={"source": "LinkedIn"}
)

In [ ]:
docsearch.add_documents(documents=[dummyDoc])

['f6a2ece2-dae3-427f-8f90-734f670a1247']

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")

In [ ]:
retrieved_docs

[Document(id='8bbaa41f-3a40-40fb-9dc8-ef557fc62979', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='9cc02f40-aecb-4762-8a6d-3fbfdc0a0529', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='3a00a4cf-960e-4500-abeb-4d682bc244b7', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [ ]:
%pip install -U langchain_google_genai

  Using cached langchain_google_genai-2.1.12-py3-none-any.whl (50 kB)
  Using cached google_ai_generativelanguage-0.11.0-py3-none-any.whl (1.4 MB)
  Using cached google_api_core-2.30.3-py3-none-any.whl (173 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl (50 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl (300 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl (14 kB)
  Using cached grpcio-1.80.0-cp39-cp39-win_amd64.whl (4.9 MB)
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.51.3
    Uninstalling grpcio-1.51.3:
      Successfully uninstalled grpcio-1.51.3
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.11.0 requires protobuf<3.20,>=3.9.2, but you have protobuf 6.33.6 which is incompatible.
tensorboard 2.11.2 requires protobuf<4,>=3.9.2, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
# from langchain_openai import ChatOpenAI

# chatModel = ChatOpenAI(model="gpt-4o")

from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

ImportError: cannot import name 'builder' from 'google.protobuf.internal' (C:\Users\DELL\AppData\Roaming\Python\Python39\site-packages\google\protobuf\internal\__init__.py)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to asnwer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use the three sentences maximum and keep the "
    "answer consice."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}